In [0]:
%run /Users/lshao10@jh.edu/ARF_NIV_Target_Trial_Emulation_DatabricksV/code/0_cohort_construction


# Script 2: Data Manipulation

This notebook constructs baseline characteristics, time-varying characteristics, and outcomes for the ARF NIV study cohort.

## Steps

1. Load configuration, clinical thresholds, source CLIF tables, and the cohort from Script 1.
2. Enrich the cohort and subset CLIF tables.
3. Construct baseline characteristics, time-varying characteristics, and outcomes.
4. Assemble the person-period dataset for downstream analyses.



#### 1. Set up and construct cohort-specific tables

Load project configuration and clinical thresholds, apply the site time zone, and check that required source tables are present.

Load the source CLIF tables, final cohort, and hospital-block key as Spark DataFrames.

Construct CLIF tables joined to the final cohort.

In [0]:
### Set up 
# --- Load required dependenies ---
import pandas as pd
import pandas.core.common as pd_common
from pandas.errors import SettingWithCopyWarning

# Compatibility workaround for comorbidipy 0.5.0
pd_common.SettingWithCopyWarning = SettingWithCopyWarning

from comorbidipy import comorbidity
import pathlib
from datetime import date, datetime
from time import perf_counter
import yaml
from pyspark.sql import functions as F

# Count elapsed time
notebook_start = perf_counter()

# --- Load local and global configurations ---
# Project root
print("Start set up...\n")

project_root = pathlib.Path.cwd()

# Load config
while not (project_root / "config").is_dir():
  if project_root.parent == project_root:
    raise Exception("Configuration folder not found\n")
  project_root = project_root.parent

with open(project_root / "config" / "config.yaml", encoding = "utf-8") as f:
  config = yaml.safe_load(f)

# Check config
required_keys = [
    "tables_location",
    "project_location",
    ## "private_tables_location",
    "institution",
    "time_zone",
    "file_type"
]

for key in required_keys:
  #  Check config key, value completeness
  if key not in config:
    raise KeyError(f"Config '{key}' not found")

  value = config[key]

  if not isinstance(value, str) or not value.strip():
    raise ValueError(f"Config '{key}' value must be a nonempty string\n")

if config["file_type"] != "parquet":
  #  Check config file type
  raise ValueError("Config 'file type' only supports parquet")

# Assign local config values
tables_location = config["tables_location"]
project_location = config["project_location"]
private_tables_location = config["private_tables_location"]
institution = config["institution"]
time_zone = config["time_zone"]

# Set site time zone as session time zone
spark.conf.set("spark.sql.session.timeZone", time_zone)

# Load and assign global config
with open(project_root / "config" / "global_config.yaml", encoding = "utf-8") as f:
  global_config = yaml.safe_load(f)

# --- Loading project related clinical thresholds ---
thresholds_path = project_root / "outlier-thresholds" / "project_outlier_thresholds.csv"

if not thresholds_path.is_file():
    raise FileNotFoundError(f"Threshold file not found here: {thresholds_path}\n")

thresholds = pd.read_csv(thresholds_path).set_index("variable_name")

# --- Load source CLIF tables ---
# Check missing tables
source_clif_table_filenames = {item.name.removesuffix("/") for item in dbutils.fs.ls(tables_location) if item.isDir()}

required_tables = {
    "clif_patient",
    "clif_hospitalization",
    "clif_vitals",
    "clif_adt",
    "clif_hospital_diagnosis",
    "clif_respiratory_support",
    "clif_medication_admin_continuous",
    "clif_labs",
    "clif_patient_assessments",
    "clif_code_status",
}

missing_table = sorted(required_tables - source_clif_table_filenames)

if len(missing_table) > 0:
  raise FileNotFoundError(f"Missing source tables: {missing_table}\n")
else: 
  print("All source tables present\n")

# Register source tables (parquet format)
source_tables = {}

for table_name in required_tables:
  source_tables[table_name] = spark.read.parquet(f"{tables_location}/{table_name}")
  print(f"{table_name} registered")

print("\nAll source tables registered\n")

# --- Load derived CLIF tables --- 
# Check missing tables
##derived_clif_table_filenames = {item.name.removesuffix("/") for item in dbutils.fs.ls(private_tables_location) if item.isDir()}

##required_derived_tables = {
##    "hospital_block_key",
##    "final_cohort"
##}

##missing_table = sorted(required_derived_tables - derived_clif_table_filenames)

##if len(missing_table) > 0:
## raise FileNotFoundError(f"Missing derived tables: {missing_table}")
##else: 
##  print("All derived tables present\n")

# Register derived tables (parquet format)
hospital_block_key = spark.table("hospital_block_key_input")
final_cohort_raw = spark.table(
    "no_pandemic_one_encounter_per_patient_input"
)

print("All derived tables registered\n")
## Replace later with:
## hospital_block_key = spark.read.csv(f"{project_location}/private_tables/hospital_block_key", header=True, inferSchema=True)
## final_cohort = spark.read.csv(f"{project_location}/private_tables/final_cohort", header=True, inferSchema=True)

print("End set up.")


In [0]:
### Constructing cohort-specific CLIF tables
# Add discharge information to the final cohort
final_cohort = final_cohort_raw.join(
    hospital_block_key.select(
        "hospital_block_id",
        F.col("block_start_admit").alias("block_start"),
        F.col("block_end_discharge").alias("block_end")
    ).distinct(),
    on="hospital_block_id",
    how="left",
)

# Construct a hospital block table for the final cohort
hospital_block_key_obj = hospital_block_key.select(
    "patient_id", "hospitalization_id", "hospital_block_id"
).join(
    final_cohort.select("hospital_block_id"), on="hospital_block_id", how="left_semi"
)

# --- Subset source CLIF tables --- 
clif_vitals = (
    source_tables["clif_vitals"]
    .filter(F.col("vital_category").isin(
        "temp_c", "heart_rate", "map", "spo2", "height_cm", "weight_kg"
    ))
    .join(hospital_block_key_obj, on="hospitalization_id", how="inner")
)

clif_adt = (
    source_tables["clif_adt"]
    .select(
        "hospitalization_id", "hospital_id", "in_dttm", "out_dttm",
        "location_name", "location_category",
    )
    .join(hospital_block_key_obj, on="hospitalization_id", how="inner")
)

clif_patient = source_tables["clif_patient"].join(
    final_cohort.select("patient_id"), on="patient_id", how="inner"
)

clif_hospitalization = source_tables["clif_hospitalization"].join(
    hospital_block_key_obj,
    on=["patient_id", "hospitalization_id"],
    how="inner",
)

clif_labs = source_tables["clif_labs"].join(
    hospital_block_key_obj, on="hospitalization_id", how="inner"
)

clif_hospital_diagnosis = source_tables["clif_hospital_diagnosis"].join(
    hospital_block_key_obj, on="hospitalization_id", how="inner"
)

clif_respiratory_support = source_tables["clif_respiratory_support"].join(
    hospital_block_key_obj, on="hospitalization_id", how="inner"
)

clif_patient_assessments = source_tables["clif_patient_assessments"].join(
    hospital_block_key_obj, on="hospitalization_id", how="inner"
)

clif_medication_admin_continuous = (
    source_tables["clif_medication_admin_continuous"]
    .join(hospital_block_key_obj, on="hospitalization_id", how="inner")
)

clif_code_status = source_tables["clif_code_status"]
